# Direct-path neural analysis: alternation vs. switch

Same data source as `change_of_mind.ipynb` (`improved_node_summary_df` + spikes), but
here we look at **direct** (`port_to_port_path_type == 'direct'`) port-to-port paths
instead of long indirect ones. Direct paths split into two kinds:

- **Alternation**: between the two ports of the same patch ([1,2], [3,4], [5,6], [7,8])
- **Switch**: between ports belonging to two different patches

For each (from-port, to-port) pair we aggregate every direct traversal of that pair,
compute each good unit's firing rate at every node along the path, and average across
traversals (mean +/- SEM). Two normalizations are used:

1. **Within a pair**: traversals of the same pair are resampled (linear interpolation
   over fractional path position) onto that pair's most common ("canonical") node
   sequence, so the x-axis can show the literal node names. In this dataset every
   traversal of a given pair already has the identical node sequence (verified below),
   so this step is a no-op here -- but it keeps the pipeline robust if that's not true
   in other sessions.
2. **Across pairs** (for the combined Alternation / Switch overview plots): pairs have
   genuinely different lengths (e.g. alternation paths = 8 nodes, switch paths = 10-11
   nodes), so those are resampled onto a shared *normalized path position* axis
   (0 = source port, 1 = destination port) instead of node identity.

In [33]:
import scipy.io
import pandas as pd
import h5py
import numpy as np
import pickle
import ast
import re
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import glob

In [34]:
TARGET_COLORS = {
    'Target1': (84/255,  64/255,  204/255),
    'Target2': (128/255, 140/255, 255/255),
    'Target3': (204/255, 115/255, 51/255),
    'Target4': (255/255, 191/255, 140/255),
    'Target5': (89/255,  153/255, 64/255),
    'Target6': (140/255, 204/255, 166/255),
    'Target7': (178/255, 102/255, 140/255),
    'Target8': (217/255, 153/255, 230/255),
}

# For switch overview plots: cycle through these linestyles per unique from_label
SWITCH_FROM_LINESTYLES = ['-', '--', ':', '-.']

## Session setup

In [35]:
# Update the session variable to match the date in your file names
session = '10/09/2025'

YY = session[-2:]
YYYY = session[-4:]
MM = session[3:5]
DD = session[:2]

DD_MM_YYYY = f"{DD}_{MM}_{YYYY}"
MM_DD_YYYY = f"{MM}_{DD}_{YYYY}"


In [36]:
spike_clusters = scipy.io.loadmat(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/aligned_spike_clusters.mat')
spike_times = scipy.io.loadmat(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/aligned_spike_times.mat')
cluster_info = pd.read_csv(rf'/var/home/almogmeir/Documents/M.Sc/Project/SpikesData/{DD}_{MM}_{YYYY}/cluster_info.tsv', sep='\t')

In [37]:
improved_node_summary_csv = f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/node_summary_df.csv"
summary_df_pkl = f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/summary_df.pkl"

with open(improved_node_summary_csv, 'rb') as f:
    improved_node_summary_df = pd.read_csv(f)

with open(summary_df_pkl, 'rb') as f:
    summary_df = pickle.load(f)

improved_node_summary_df.shape

(5858, 20)

## Spike data

In [38]:
good_units = cluster_info[cluster_info['group'] == 'good'].copy().reset_index(drop=True)
mua_units  = cluster_info[cluster_info['group'] == 'mua'].copy().reset_index(drop=True)
# noise units are dropped entirely

print(f"Total clusters : {len(cluster_info)}")
print(f"Good units     : {len(good_units)}")
print(f"MUA units      : {len(mua_units)}")
print(f"Noise dropped  : {(cluster_info['group'] == 'noise').sum()}")

Total clusters : 1005
Good units     : 34
MUA units      : 137
Noise dropped  : 793


In [39]:
FPS = 40       # video frames per second
FS = 30_000    # Neuropixels sampling rate (Hz)

def _unwrap_mat(d):
    keys = [k for k in d if not k.startswith('__')]
    arr = d[keys[0]]
    return arr.flatten()

spike_times_all    = _unwrap_mat(spike_times)
spike_clusters_all = _unwrap_mat(spike_clusters)

# Maze trials are 31-766 (1-indexed) -> indices 30:767 (0-indexed, 737 trials)
MAZE_START = 30
MAZE_END   = 767

maze_spike_times    = spike_times_all[MAZE_START:MAZE_END]
maze_spike_clusters = spike_clusters_all[MAZE_START:MAZE_END]

assert len(maze_spike_times) == 737, f"Expected 737 maze trials, got {len(maze_spike_times)}"
print(f"Maze trials extracted: {len(maze_spike_times)} (trials 31-766)")

Maze trials extracted: 737 (trials 31-766)


In [40]:
# summary_df has MultiIndex columns (from the tracking pipeline); flatten the
# behavior-only columns we need to build a trial -> absolute-time lookup.
sdf = summary_df.copy()
sdf.columns = ['_'.join([c for c in col if c]) if isinstance(col, tuple) else col for col in sdf.columns]

trial_meta = sdf.groupby('trial_idx').agg(
    start_frame=('frame_idx_global', 'min'),
    trial_length=('trial_length', 'first'),
)
trial_meta['start_abs'] = trial_meta['start_frame'] / FPS
trial_meta['end_abs'] = trial_meta['start_abs'] + trial_meta['trial_length']

def trials_overlapping(t_lo, t_hi):
    """All pipeline 'trial' indices whose time range overlaps [t_lo, t_hi]."""
    m = (trial_meta['end_abs'] >= t_lo) & (trial_meta['start_abs'] <= t_hi)
    return trial_meta.index[m].tolist()

def gather_relative_spikes(trials_in_path, cluster_id, t0_abs):
    """Spikes for one cluster across the given trials, as seconds relative to t0_abs,
    by stitching each trial's spike times (stored relative to that trial's own start, in
    samples) onto the global video clock via trial_meta['start_abs']."""
    rel_chunks = []
    for t in trials_in_path:
        spike_t = np.asarray(maze_spike_times[t]).flatten() / FS
        spike_c = np.asarray(maze_spike_clusters[t]).flatten()
        abs_t = trial_meta.loc[t, 'start_abs'] + spike_t
        rel_chunks.append(abs_t[spike_c == cluster_id] - t0_abs)
    return np.concatenate(rel_chunks) if rel_chunks else np.array([])

## Step 1: Extract direct-path instances

One instance per contiguous run of a `path_pair_label` whose `port_to_port_path_type`
is `'direct'`. Each instance carries its full node sequence plus per-node start/end
times. `('TargetN', 'TargetM')` pairs are classified `alternation` if N and M are in
the same port-pair group ([1,2],[3,4],[5,6],[7,8]) and `switch` otherwise.

`start_frame`/`end_frame` in `improved_node_summary_df` are `frame_idx_global` values
(verified against `summary_df`), so they're trustworthy, trial-time-aligned frame
indices -- per-node timing here always derives from them directly:
`start_abs = start_frame / FPS`, `end_abs = (end_frame + 1) / FPS`. The stored
`duration_frames` column is **not** used: per `FRAME_OFFSET_BUG_ANALYSIS.md` /
`FRAME_OFFSET_FIX.md`, the upstream traversal-reconstruction step can steal frames
between adjacent runs when inferring a missing node, which desyncs `duration_frames`
(list-length-based) from the true `start_frame`/`end_frame` span for those rows --
`end_frame - start_frame + 1` is the reliable duration.

In [41]:
def parse_pair_label(label):
    try:
        return ast.literal_eval(label)
    except (ValueError, SyntaxError, TypeError):
        return (None, None)

def target_group(label):
    """Group index (0-3) for a 'TargetN' label: ports [1,2]->0, [3,4]->1, [5,6]->2, [7,8]->3."""
    n = int(re.search(r'\d+', str(label)).group())
    return (n - 1) // 2, n

df_sorted = improved_node_summary_df.sort_values('node_visit_idx').reset_index(drop=True)

# One row per contiguous run of identical path_pair_label (the row marking the start
# of each port-to-port path traversal).
path_segments = []
prev_label = None
for _, row in df_sorted.iterrows():
    label = row['path_pair_label']
    if pd.isna(label):
        prev_label = None
        continue
    if label != prev_label:
        path_segments.append(row)
    prev_label = label

print(f"Total port-to-port path segments in session: {len(path_segments)}")

direct_instances = []
for seg_row in path_segments:
    if seg_row['port_to_port_path_type'] != 'direct':
        continue

    start_nv = int(seg_row['node_visit_idx'])
    seq_len = int(seg_row['path_seq_length_unique'])
    end_nv = start_nv + seq_len

    seq_rows = improved_node_summary_df[
        (improved_node_summary_df['node_visit_idx'] >= start_nv) &
        (improved_node_summary_df['node_visit_idx'] <= end_nv)
    ].sort_values('node_visit_idx').reset_index(drop=True)

    if seq_rows.empty:
        continue

    from_label, to_label = parse_pair_label(seg_row['path_pair_label'])
    from_grp, _ = target_group(from_label)
    to_grp, _ = target_group(to_label)
    path_class = 'alternation' if from_grp == to_grp else 'switch'

    # Reward received at source port during its dwell
    source_reward = float(seq_rows.iloc[0].get('reward_size_during_node', 0) or 0)
    rewarded = source_reward > 0

    node_seq = seq_rows['node_name'].tolist()
    start_frames = seq_rows['start_frame'].to_numpy()
    end_frames = seq_rows['end_frame'].to_numpy()
    starts_abs = start_frames / FPS
    ends_abs = (end_frames + 1) / FPS
    durations_sec = ends_abs - starts_abs

    direct_instances.append({
        'pair_label': (from_label, to_label),
        'path_class': path_class,
        'rewarded': rewarded,
        'trial_idx': int(seg_row['trial_idx']),
        'node_seq': node_seq,
        'n_nodes': len(node_seq),
        'starts_abs': starts_abs,
        'ends_abs': ends_abs,
        'durations_sec': durations_sec,
        'path_start_abs': starts_abs[0],
        'path_end_abs': ends_abs[-1],
        # Raw global frame boundaries of the whole path (first node's start_frame,
        # last node's end_frame) -- used by the edge-inclusive analysis below to slice
        # the matching rows out of traversal_df, which has no direct/indirect labeling
        # of its own.
        'path_frame_start': int(start_frames[0]),
        'path_frame_end': int(end_frames[-1]),
    })

print(f"Direct path instances: {len(direct_instances)}")

pairs_to_instances = {}
for inst in direct_instances:
    pairs_to_instances.setdefault(inst['pair_label'], []).append(inst)

print(f"Unique direct (from, to) pairs: {len(pairs_to_instances)}")

Total port-to-port path segments in session: 521
Direct path instances: 426
Unique direct (from, to) pairs: 30


In [42]:
pairs_to_rewarded_instances = {}
pairs_to_unrewarded_instances = {}
for inst in direct_instances:
    d = pairs_to_rewarded_instances if inst['rewarded'] else pairs_to_unrewarded_instances
    d.setdefault(inst['pair_label'], []).append(inst)

n_rew   = sum(1 for i in direct_instances if i['rewarded'])
n_unrew = sum(1 for i in direct_instances if not i['rewarded'])
print(f"Rewarded source-port instances  : {n_rew}")
print(f"Unrewarded source-port instances: {n_unrew}")

Rewarded source-port instances  : 245
Unrewarded source-port instances: 181


In [43]:
# Sanity check: how many pairs have traversals of inconsistent node-sequence length?
pair_overview_rows = []
for pair_label, instances in pairs_to_instances.items():
    lengths = sorted(set(inst['n_nodes'] for inst in instances))
    pair_overview_rows.append({
        'pair_label': pair_label,
        'path_class': instances[0]['path_class'],
        'n_instances': len(instances),
        'unique_lengths': lengths,
    })

pair_overview_df = pd.DataFrame(pair_overview_rows).sort_values('n_instances', ascending=False).reset_index(drop=True)
n_inconsistent = (pair_overview_df['unique_lengths'].apply(len) > 1).sum()
print(f"Pairs with more than one observed node-sequence length: {n_inconsistent} (resampling handles these if present)")
display(pair_overview_df)

Pairs with more than one observed node-sequence length: 2 (resampling handles these if present)


,pair_label,path_class,n_instances,unique_lengths
0,"(Target4, Target3)",alternation,56,[9]
1,"(Target7, Target8)",alternation,53,[9]
2,"(Target8, Target7)",alternation,52,[9]
3,"(Target2, Target1)",alternation,47,[9]
4,"(Target3, Target4)",alternation,46,[9]
5,"(Target5, Target6)",alternation,38,[9]
6,"(Target6, Target5)",alternation,36,"[3, 9]"
7,"(Target1, Target2)",alternation,30,"[8, 9]"
8,"(Target2, Target8)",switch,12,[11]
9,"(Target5, Target4)",switch,7,[11]


## Step 2: Per-node firing rate + length normalization helpers

`compute_node_rates` gives one firing-rate value (Hz) per node visited along a single
path instance, for one unit. `resample_to_grid` linearly interpolates any such sequence
(or its mean/SEM) from its own length onto a target number of points, indexed by
fractional position along the path (0 = source port, 1 = destination port). This is
the single normalization primitive used both for aligning same-pair instances onto a
canonical node grid, and for aligning different-length pairs onto the shared overview
axis.

In [44]:
MIN_INSTANCES = 2  # need at least 2 traversals to compute a meaningful mean +/- SEM

def compute_node_rates(instance, cluster_id):
    """Firing rate (Hz) of one unit during each node-visit of one path instance."""
    trials_in_path = trials_overlapping(instance['path_start_abs'], instance['path_end_abs'])
    rel_spikes = gather_relative_spikes(trials_in_path, cluster_id, instance['path_start_abs'])
    rel_starts = instance['starts_abs'] - instance['path_start_abs']
    rel_ends = instance['ends_abs'] - instance['path_start_abs']

    rates = np.empty(instance['n_nodes'])
    for i in range(instance['n_nodes']):
        count = np.sum((rel_spikes >= rel_starts[i]) & (rel_spikes < rel_ends[i]))
        rates[i] = count / instance['durations_sec'][i]
    return rates

def resample_to_grid(values, n_grid):
    """Linearly interpolate `values` (length L) onto `n_grid` points over fractional
    position 0..1."""
    values = np.asarray(values, dtype=float)
    L = len(values)
    if L == 1:
        return np.full(n_grid, values[0])
    src_frac = np.linspace(0, 1, L)
    dst_frac = np.linspace(0, 1, n_grid)
    return np.interp(dst_frac, src_frac, values)

def canonical_sequence(instances):
    """Most common exact node sequence among a pair's traversals."""
    seqs = [tuple(inst['node_seq']) for inst in instances]
    most_common_seq, _ = Counter(seqs).most_common(1)[0]
    return list(most_common_seq)

## Step 3: Aggregate mean +/- SEM per (pair, good unit)

For every direct-path pair with at least `MIN_INSTANCES` traversals, and every good
unit, resample each traversal's per-node rate onto the pair's canonical node grid, then
take the mean and SEM across traversals. Cached in `pair_unit_stats` for reuse by both
the per-pair plots and the Alternation/Switch overview plots.

In [45]:
pair_unit_stats = {}  # (pair_label, cluster_id) -> dict(mean, sem, n, canonical_seq, path_class)
skipped_pairs = []

good_cluster_ids = good_units['cluster_id'].astype(int).tolist()

for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        skipped_pairs.append((pair_label, len(instances)))
        continue

    canonical_seq = canonical_sequence(instances)
    L = len(canonical_seq)
    path_class = instances[0]['path_class']

    for cid in good_cluster_ids:
        resampled = np.vstack([
            resample_to_grid(compute_node_rates(inst, cid), L)
            for inst in instances
        ])
        mean = resampled.mean(axis=0)
        sem = resampled.std(axis=0, ddof=1) / np.sqrt(resampled.shape[0])
        pair_unit_stats[(pair_label, cid)] = {
            'mean': mean,
            'sem': sem,
            'n': len(instances),
            'canonical_seq': canonical_seq,
            'path_class': path_class,
        }

print(f"Computed stats for {len(pair_unit_stats) // max(len(good_cluster_ids), 1)} pairs x {len(good_cluster_ids)} good units")
print(f"Pairs skipped (fewer than {MIN_INSTANCES} traversals): {len(skipped_pairs)}")
for pl, n in skipped_pairs:
    print(f"  {pl}: n={n}")

def _compute_node_pair_stats(split_dict):
    stats = {}
    for pair_label, instances in split_dict.items():
        if len(instances) < MIN_INSTANCES:
            continue
        canonical_seq = canonical_sequence(instances)
        L = len(canonical_seq)
        path_class = instances[0]['path_class']
        for cid in good_cluster_ids:
            resampled = np.vstack([
                resample_to_grid(compute_node_rates(inst, cid), L)
                for inst in instances
            ])
            mean = resampled.mean(axis=0)
            sem = resampled.std(axis=0, ddof=1) / np.sqrt(resampled.shape[0])
            stats[(pair_label, cid)] = {
                'mean': mean, 'sem': sem, 'n': len(instances),
                'canonical_seq': canonical_seq, 'path_class': path_class,
            }
    return stats

pair_unit_stats_rewarded   = _compute_node_pair_stats(pairs_to_rewarded_instances)
pair_unit_stats_unrewarded = _compute_node_pair_stats(pairs_to_unrewarded_instances)
n_rew_pairs   = len(pair_unit_stats_rewarded)   // max(len(good_cluster_ids), 1)
n_unrew_pairs = len(pair_unit_stats_unrewarded) // max(len(good_cluster_ids), 1)
print(f"Node stats — Rewarded pairs: {n_rew_pairs} | Unrewarded pairs: {n_unrew_pairs}")

Computed stats for 22 pairs x 34 good units
Pairs skipped (fewer than 2 traversals): 8
  ('Target8', 'Target6'): n=1
  ('Target8', 'Target1'): n=1
  ('Target1', 'Target5'): n=1
  ('Target4', 'Target6'): n=1
  ('Target4', 'Target1'): n=1
  ('Target2', 'Target7'): n=1
  ('Target2', 'Target5'): n=1
  ('Target4', 'Target5'): n=1
Node stats — Rewarded pairs: 10 | Unrewarded pairs: 22


## Step 4: Per-pair plots

One figure per (pair, good unit): mean firing rate (+/- SEM shaded band) at each node
along the canonical path, x-axis labeled with the actual node names. Saved under
`resources/outputs/{session}/direct_path_neural/{alternation|switch}/{from}_to_{to}/unit_{cluster_id}.png`.

In [46]:
output_root = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural")
output_root.mkdir(parents=True, exist_ok=True)

n_saved = 0
for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue

    from_label, to_label = pair_label
    path_class = instances[0]['path_class']
    pair_dir = output_root / path_class / f"{from_label}_to_{to_label}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    for cid in good_cluster_ids:
        c = TARGET_COLORS.get(to_label, 'steelblue')
        # Use all-instances canonical sequence for shared x-axis width hint
        L_all = len(pair_unit_stats[(pair_label, cid)]['canonical_seq'])

        fig, (ax_rew, ax_unrew) = plt.subplots(1, 2, figsize=(max(10, L_all * 0.9), 4))

        for ax, split_stats, split_label in [
            (ax_rew,   pair_unit_stats_rewarded.get((pair_label, cid)),   'Rewarded'),
            (ax_unrew, pair_unit_stats_unrewarded.get((pair_label, cid)), 'Unrewarded'),
        ]:
            ax.set_title(split_label, fontsize=9)
            if split_stats is None:
                ax.text(0.5, 0.5, f'< {MIN_INSTANCES} traversals', ha='center', va='center',
                        transform=ax.transAxes, fontsize=9, color='gray')
                ax.set_xticks([])
            else:
                mean, sem, n = split_stats['mean'], split_stats['sem'], split_stats['n']
                seq = split_stats['canonical_seq']
                x = np.arange(len(seq))
                ax.plot(x, mean, color=c, linewidth=1.8)
                ax.fill_between(x, mean - sem, mean + sem, color=c, alpha=0.3, label=f'SEM (n={n})')
                ax.set_xticks(x)
                ax.set_xticklabels(seq, rotation=90, fontsize=7)
                ax.legend(fontsize=7, loc='upper right')
            ax.set_xlabel('Path node (canonical sequence)')
            ax.set_ylabel('Firing rate (Hz)')

        fig.suptitle(
            f"Unit {cid} | {path_class}: {from_label} -> {to_label}",
            fontsize=9,
        )
        fig.tight_layout()
        fig.savefig(pair_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

print(f"Saved {n_saved} per-pair plots under {output_root}")

Saved 748 per-pair plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural


## Step 5: Alternation / Switch overview plots

One figure per good unit per path class, overlaying every pair of that class as its own
line (+/- SEM band), on a shared **normalized path-position** axis (0 = source port,
1 = destination port) since pairs of the same class can still have different lengths
(e.g. alternation paths in this session are all 8 nodes, but switch paths are 10-11 --
the normalized axis is what makes overlaying them meaningful). Saved under
`resources/outputs/{session}/direct_path_neural/{alternation|switch}_overview/unit_{cluster_id}.png`.

In [47]:
OVERVIEW_GRID = 100
frac_grid = np.linspace(0, 1, OVERVIEW_GRID)

for path_class in ['alternation', 'switch']:
    class_dir = output_root / f"{path_class}_overview"
    class_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]

    # For alternation, use a node-indexed x-axis if all paths share the same canonical length
    use_node_indexed = False
    node_x_vals = node_x_tick_positions = node_x_tick_labels = None
    if path_class == 'alternation' and relevant_pairs:
        lengths = set(len(pair_unit_stats[(pl, good_cluster_ids[0])]['canonical_seq']) for pl in relevant_pairs)
        if len(lengths) == 1:
            L_shared = lengths.pop()
            mid = (L_shared - 1) // 2
            node_x_vals = np.arange(L_shared)
            node_x_tick_positions = [0, mid, L_shared - 1]
            node_x_tick_labels = ['Source\nPort', 'Midpoint', 'Target\nPort']
            use_node_indexed = True

    # For switch, assign a linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    n_saved = 0
    for cid in good_cluster_ids:
        fig, (ax_rew, ax_unrew) = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

        for ax, split_stats_dict, split_label in [
            (ax_rew,   pair_unit_stats_rewarded,   'Rewarded'),
            (ax_unrew, pair_unit_stats_unrewarded, 'Unrewarded'),
        ]:
            ax.set_title(split_label, fontsize=10)
            for pair_label in relevant_pairs:
                if (pair_label, cid) not in split_stats_dict:
                    continue
                stats = split_stats_dict[(pair_label, cid)]
                from_label, to_label = pair_label
                color = TARGET_COLORS.get(to_label, 'gray')
                ls = from_to_ls.get(from_label, '-')

                if use_node_indexed:
                    mean_plot = stats['mean']
                    sem_plot  = stats['sem']
                    x_plot    = node_x_vals
                else:
                    L = len(stats['canonical_seq'])
                    mean_plot = resample_to_grid(stats['mean'], OVERVIEW_GRID) if L != OVERVIEW_GRID else stats['mean']
                    sem_plot  = resample_to_grid(stats['sem'],  OVERVIEW_GRID) if L != OVERVIEW_GRID else stats['sem']
                    x_plot    = frac_grid

                ax.plot(x_plot, mean_plot, color=color, linewidth=1.5, linestyle=ls,
                         label=f"{from_label}->{to_label} (n={stats['n']})")
                ax.fill_between(x_plot, mean_plot - sem_plot, mean_plot + sem_plot,
                                 color=color, alpha=0.15)

            if use_node_indexed:
                ax.set_xticks(node_x_tick_positions)
                ax.set_xticklabels(node_x_tick_labels)
                ax.set_xlabel('Path node position')
            else:
                ax.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')
            ax.legend(fontsize=6, loc='upper right', ncol=2)

        ax_rew.set_ylabel('Firing rate (Hz)')
        fig.suptitle(f"Unit {cid} | {path_class.capitalize()} direct paths", fontsize=10)
        fig.tight_layout()
        fig.savefig(class_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} '{path_class}' overview plots under {class_dir} ({len(relevant_pairs)} pairs overlaid)")

Saved 34 'alternation' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural/alternation_overview (8 pairs overlaid)
Saved 34 'switch' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural/switch_overview (14 pairs overlaid)


## Save pair summary

In [48]:
summary_out_rows = []
for pair_label, instances in pairs_to_instances.items():
    from_label, to_label = pair_label
    summary_out_rows.append({
        'from_port': from_label,
        'to_port': to_label,
        'path_class': instances[0]['path_class'],
        'n_instances': len(instances),
        'n_nodes': instances[0]['n_nodes'],
        'used_in_aggregation': len(instances) >= MIN_INSTANCES,
        'canonical_sequence': ' -> '.join(canonical_sequence(instances)),
    })

direct_path_pair_summary_df = pd.DataFrame(summary_out_rows).sort_values('n_instances', ascending=False).reset_index(drop=True)
out_csv = output_root / 'direct_path_pair_summary.csv'
direct_path_pair_summary_df.to_csv(out_csv, index=False)
print(f"Saved pair summary to {out_csv}")
display(direct_path_pair_summary_df)

Saved pair summary to /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural/direct_path_pair_summary.csv


,from_port,to_port,path_class,n_instances,n_nodes,used_in_aggregation,canonical_sequence
0,Target4,Target3,alternation,56,9,True,L510 -> L45 -> L32 -> L21 -> L10 -> L20 -> L31...
1,Target7,Target8,alternation,53,9,True,R59 -> R44 -> R32 -> R21 -> R10 -> R20 -> R31 ...
2,Target8,Target7,alternation,52,9,True,R55 -> R42 -> R31 -> R20 -> R10 -> R21 -> R32 ...
3,Target2,Target1,alternation,47,9,True,R526 -> R413 -> R36 -> R23 -> R11 -> R22 -> R3...
4,Target3,Target4,alternation,46,9,True,L56 -> L43 -> L31 -> L20 -> L10 -> L21 -> L32 ...
5,Target5,Target6,alternation,38,9,True,L521 -> L410 -> L35 -> L22 -> L11 -> L23 -> L3...
6,Target6,Target5,alternation,36,9,True,L525 -> L412 -> L36 -> L23 -> L11 -> L22 -> L3...
7,Target1,Target2,alternation,30,9,True,R522 -> R411 -> R35 -> R22 -> R11 -> R23 -> R3...
8,Target2,Target8,switch,12,11,True,R526 -> R413 -> R36 -> R23 -> R11 -> R0 -> R10...
9,Target5,Target4,switch,7,11,True,L521 -> L410 -> L35 -> L22 -> L11 -> L0 -> L10...


# Additional analysis: continuous-time version (node-agnostic)

The per-node analysis above only counts spikes while the mouse is parked *at* a node
-- it has no data during the "edge" transit gaps between consecutive node-visits, so
those gaps are silently skipped rather than contributing zero/low/high firing.

This version instead treats each path traversal as one continuous time window:
`start = first node's start_frame`, `end = last node's end_frame` (i.e. exactly
`path_start_abs` / `path_end_abs`, already computed above), and bins spikes across
the *entire* window at a fixed bin width, with no reference to node identity at all.

Unlike the per-node version, normalization here is *not* a no-op: even though every
traversal of a given pair visits the same fixed number of nodes, the total elapsed
time for the path (dwell times + transit gaps) still varies traversal to traversal.
So every instance's binned rate trace is resampled (same `resample_to_grid` helper as
above) onto a shared **normalized path-time** axis (0 = path start, 1 = path end)
before averaging across traversals of a pair, and again across pairs for the
Alternation/Switch overview.

This is purely additive -- it doesn't replace the per-node analysis, and writes to a
separate output folder (`direct_path_neural_continuous/`).

In [49]:
from scipy.ndimage import gaussian_filter1d

BIN_SIZE_CONTINUOUS = 0.05      # seconds, histogram bin width for the raw rate
SMOOTH_SIGMA_CONTINUOUS = 0.15  # seconds, stdev of the Gaussian smoothing kernel
CONTINUOUS_GRID = 100           # points on the shared normalized-path-time axis

def compute_continuous_rate(instance, cluster_id):
    """Binned + lightly smoothed firing rate (Hz) of one unit across the path's
    full time window (path_start_abs -> path_end_abs), independent of node identity."""
    duration = instance['path_end_abs'] - instance['path_start_abs']
    n_bins = max(1, round(duration / BIN_SIZE_CONTINUOUS))
    bin_edges = np.linspace(0, duration, n_bins + 1)
    bin_width = bin_edges[1] - bin_edges[0]

    trials_in_path = trials_overlapping(instance['path_start_abs'], instance['path_end_abs'])
    rel_spikes = gather_relative_spikes(trials_in_path, cluster_id, instance['path_start_abs'])

    counts, _ = np.histogram(rel_spikes, bins=bin_edges)
    rate = counts.astype(float) / bin_width
    sigma_bins = SMOOTH_SIGMA_CONTINUOUS / bin_width
    return gaussian_filter1d(rate, sigma=sigma_bins, mode='constant')

## Aggregate mean +/- SEM per (pair, good unit), continuous-time version

Same `MIN_INSTANCES` threshold and pair grouping as the per-node analysis -- only the
within-pair resampling target changes (a fixed `CONTINUOUS_GRID` of normalized-time
points instead of the pair's canonical node count).

In [50]:
continuous_pair_unit_stats = {}  # (pair_label, cluster_id) -> dict(mean, sem, n, mean_duration, path_class)

for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue

    path_class = instances[0]['path_class']
    mean_duration = np.mean([inst['path_end_abs'] - inst['path_start_abs'] for inst in instances])

    for cid in good_cluster_ids:
        resampled = np.vstack([
            resample_to_grid(compute_continuous_rate(inst, cid), CONTINUOUS_GRID)
            for inst in instances
        ])
        mean = resampled.mean(axis=0)
        sem = resampled.std(axis=0, ddof=1) / np.sqrt(resampled.shape[0])
        continuous_pair_unit_stats[(pair_label, cid)] = {
            'mean': mean,
            'sem': sem,
            'n': len(instances),
            'mean_duration': mean_duration,
            'path_class': path_class,
        }

print(f"Computed continuous-time stats for {len(continuous_pair_unit_stats) // max(len(good_cluster_ids), 1)} pairs x {len(good_cluster_ids)} good units")

def _compute_continuous_pair_stats(split_dict):
    stats = {}
    for pair_label, instances in split_dict.items():
        if len(instances) < MIN_INSTANCES:
            continue
        path_class = instances[0]['path_class']
        mean_duration = np.mean([inst['path_end_abs'] - inst['path_start_abs'] for inst in instances])
        for cid in good_cluster_ids:
            resampled = np.vstack([
                resample_to_grid(compute_continuous_rate(inst, cid), CONTINUOUS_GRID)
                for inst in instances
            ])
            mean = resampled.mean(axis=0)
            sem = resampled.std(axis=0, ddof=1) / np.sqrt(resampled.shape[0])
            stats[(pair_label, cid)] = {
                'mean': mean, 'sem': sem, 'n': len(instances),
                'mean_duration': mean_duration, 'path_class': path_class,
            }
    return stats

continuous_pair_unit_stats_rewarded   = _compute_continuous_pair_stats(pairs_to_rewarded_instances)
continuous_pair_unit_stats_unrewarded = _compute_continuous_pair_stats(pairs_to_unrewarded_instances)
n_rew   = len(continuous_pair_unit_stats_rewarded)   // max(len(good_cluster_ids), 1)
n_unrew = len(continuous_pair_unit_stats_unrewarded) // max(len(good_cluster_ids), 1)
print(f"Continuous stats — Rewarded pairs: {n_rew} | Unrewarded pairs: {n_unrew}")

Computed continuous-time stats for 22 pairs x 34 good units
Continuous stats — Rewarded pairs: 10 | Unrewarded pairs: 22


## Per-pair plots (continuous-time)

X-axis is normalized path time (0 = path start, 1 = path end) since, unlike the
per-node version, raw seconds aren't comparable across traversals of the same pair.
The mean traversal duration is noted in the title for context. Saved under
`resources/outputs/{session}/direct_path_neural_continuous/{alternation|switch}/{from}_to_{to}/unit_{cluster_id}.png`.

In [51]:
continuous_output_root = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural_continuous")
continuous_output_root.mkdir(parents=True, exist_ok=True)

continuous_frac_grid = np.linspace(0, 1, CONTINUOUS_GRID)

n_saved = 0
for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue

    from_label, to_label = pair_label
    path_class = instances[0]['path_class']
    pair_dir = continuous_output_root / path_class / f"{from_label}_to_{to_label}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    for cid in good_cluster_ids:
        c = TARGET_COLORS.get(to_label, 'darkorange')

        fig, (ax_rew, ax_unrew) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

        for ax, split_stats, split_label in [
            (ax_rew,   continuous_pair_unit_stats_rewarded.get((pair_label, cid)),   'Rewarded'),
            (ax_unrew, continuous_pair_unit_stats_unrewarded.get((pair_label, cid)), 'Unrewarded'),
        ]:
            ax.set_title(split_label, fontsize=9)
            if split_stats is None:
                ax.text(0.5, 0.5, f'< {MIN_INSTANCES} traversals', ha='center', va='center',
                        transform=ax.transAxes, fontsize=9, color='gray')
            else:
                mean, sem, n = split_stats['mean'], split_stats['sem'], split_stats['n']
                ax.plot(continuous_frac_grid, mean, color=c, linewidth=1.8)
                ax.fill_between(continuous_frac_grid, mean - sem, mean + sem, color=c, alpha=0.1,
                                label=f'SEM (n={n}, {split_stats["mean_duration"]:.2f}s)')
                ax.legend(fontsize=7, loc='upper right')
            ax.set_xlabel('Normalized path time (0 = source port start, 1 = target port end)')

        ax_rew.set_ylabel('Firing rate (Hz)')
        fig.suptitle(
            f"Unit {cid} | {path_class}: {from_label} -> {to_label}",
            fontsize=9,
        )
        fig.tight_layout()
        fig.savefig(pair_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

print(f"Saved {n_saved} per-pair continuous-time plots under {continuous_output_root}")

Saved 748 per-pair continuous-time plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_continuous


## Alternation / Switch overview plots (continuous-time)

Same idea as the node-based overview: overlay every pair of a class on the shared
normalized-time axis, one figure per good unit. Saved under
`resources/outputs/{session}/direct_path_neural_continuous/{alternation|switch}_overview/unit_{cluster_id}.png`.

In [52]:
for path_class in ['alternation', 'switch']:
    class_dir = continuous_output_root / f"{path_class}_overview"
    class_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]

    # For switch, assign a linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    n_saved = 0
    for cid in good_cluster_ids:
        fig, (ax_rew, ax_unrew) = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

        for ax, split_stats_dict, split_label in [
            (ax_rew,   continuous_pair_unit_stats_rewarded,   'Rewarded'),
            (ax_unrew, continuous_pair_unit_stats_unrewarded, 'Unrewarded'),
        ]:
            ax.set_title(split_label, fontsize=10)
            for pair_label in relevant_pairs:
                if (pair_label, cid) not in split_stats_dict:
                    continue
                stats = split_stats_dict[(pair_label, cid)]
                from_label, to_label = pair_label
                color = TARGET_COLORS.get(to_label, 'gray')
                ls = from_to_ls.get(from_label, '-')
                mean, sem = stats['mean'], stats['sem']

                ax.plot(continuous_frac_grid, mean, color=color, linewidth=1.5, linestyle=ls,
                         label=f"{from_label}->{to_label} (n={stats['n']}, {stats['mean_duration']:.1f}s)")
                ax.fill_between(continuous_frac_grid, mean - sem, mean + sem, color=color, alpha=0.1)

            ax.set_xlabel('Normalized path time (0 = source port start, 1 = target port end)')
            ax.legend(fontsize=6, loc='upper right', ncol=2)

        ax_rew.set_ylabel('Firing rate (Hz)')
        fig.suptitle(f"Unit {cid} | {path_class.capitalize()} direct paths (continuous-time)", fontsize=10)
        fig.tight_layout()
        fig.savefig(class_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} '{path_class}' continuous-time overview plots under {class_dir} ({len(relevant_pairs)} pairs overlaid)")

Saved 34 'alternation' continuous-time overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_continuous/alternation_overview (8 pairs overlaid)
Saved 34 'switch' continuous-time overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_continuous/switch_overview (14 pairs overlaid)


# Additional analysis: full node + edge sequence version

The per-node version (Steps 1-5) only has firing rate at node dwell-points; the
continuous-time version captures the transit gaps too but discards element identity
entirely. This version keeps *both*: the literal node-and-edge sequence the mouse
walked, firing rate computed separately for every node dwell **and** every edge
transit.

The fine-grained node+edge traversal lives in `traversal_df.csv` (same outputs folder
as `node_summary_df.csv`), but it has no `direct`/`indirect`/pair-label columns of its
own -- the path identification only happens in `improved_node_summary_df`. So rather
than re-deriving path boundaries from scratch, we reuse the direct-path instances
already extracted in Step 1 (`direct_instances` / `pairs_to_instances`) purely for
their **frame-range boundaries** (`path_frame_start`, `path_frame_end`, the first
node's `start_frame` and the last node's `end_frame`), and slice every `traversal_df`
row -- node *and* edge -- whose own frame range falls inside that span. That's the
"matching" step: it happens once per path instance via a frame-range filter, not
row-by-row.

One consequence: instance length is no longer fixed even within a pair (small
untracked single-frame gaps or brief node-edge-node wobbles add/remove elements
between traversals), so the same canonical-sequence + fractional-position resampling
used elsewhere is doing real work here.

In [53]:
# Load node_edge_df.csv — the bounce-fixed, path-annotated full traversal.
# This file is generated by the same pipeline run as node_summary_df.csv
# (via align_frames_trials.ipynb → build_node_edge_df), so frame boundaries
# are guaranteed to be in perfect sync.
node_edge_csv = (
    f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/"
    f"resources/outputs/{DD}_{MM}_{YYYY}/node_edge_df.csv"
)
node_edge_df = pd.read_csv(node_edge_csv).sort_values('start_frame_global').reset_index(drop=True)

print(f"node_edge_df rows (nodes + edges): {len(node_edge_df)}")
print(f"  node rows : {(node_edge_df['location_type'] == 'node').sum()}")
print(f"  edge rows : {(node_edge_df['location_type'] == 'edge').sum()}")
print()

# Sanity: both files come from the same pipeline run — frame ranges must match.
ne_max = node_edge_df['end_frame_global'].max()
ns_max = improved_node_summary_df['end_frame'].max()
ne_tri = node_edge_df['trial_idx'].max()
ns_tri = improved_node_summary_df['trial_idx'].max()
print(f"node_edge_df  max end_frame_global : {ne_max}")
print(f"node_summary  max end_frame        : {ns_max}  {'✓ match' if ne_max == ns_max else '✗ MISMATCH — re-run align_frames_trials.ipynb'}")
print(f"node_edge_df  max trial_idx        : {ne_tri}")
print(f"node_summary  max trial_idx        : {ns_tri}  {'✓ match' if ne_tri == ns_tri else '✗ MISMATCH'}")
assert ne_max == ns_max, (
    "end_frame mismatch between node_edge_df and node_summary_df — "
    "re-run align_frames_trials.ipynb to regenerate both from the same pipeline."
)


node_edge_df rows (nodes + edges): 10091
  node rows : 6129
  edge rows : 3962

node_edge_df  max end_frame_global : 145416
node_summary  max end_frame        : 145416  ✓ match
node_edge_df  max trial_idx        : 736
node_summary  max trial_idx        : 736  ✓ match


## Build the matched full element sequence per direct-path instance

For each instance already in `direct_instances`, slice every `node_edge_df` row whose
`[start_frame_global, end_frame_global]` falls within `[path_frame_start, path_frame_end]`.

**Expected structure for every direct path:**  
- **9 nodes** (source port, 7 internal nodes, destination port)  
- **6 physical edges** (the 2 leaf-to-parent connections are virtual and produce no edge row)  
- **= 15 elements total**

**Port-sharing guarantee:** port Y appears in *both* the X→Y and Y→Z path sequences
because the filter is `>= start_frame[X]` (or `>= start_frame[Y]`) and `<= end_frame[Y]`
(or `<= end_frame[Z]`), so port Y's row satisfies both ranges exactly.

Results are stored back onto the same instance dicts (new keys only), so
`pairs_to_instances` automatically picks them up.


In [54]:
def edge_label_str(raw):
    """'(\'L31\', \'L43\')' -> 'L31-L43'"""
    try:
        a, b = ast.literal_eval(raw)
        return f"{a}-{b}"
    except (ValueError, SyntaxError, TypeError):
        return str(raw)


def build_full_element_sequence(instance):
    """Every node_edge_df row (node or edge) inside the instance's frame span.

    Both the source and destination port boundaries are looked up directly in
    node_edge_df (with a ±20 frame tolerance) rather than using the
    path_frame_start / path_frame_end values from improved_node_summary_df.
    This is necessary because node_edge_df is built before the reward-frame fix
    in align_frames_trials, so port-visit boundaries can differ slightly between
    the two files — the reward-fix can shift path_frame_start either earlier or
    later than the source port's actual start_frame_global in node_edge_df,
    causing the source node row to be excluded or spurious preceding rows to be
    included.
    """
    frame_start = instance['path_frame_start']
    frame_end   = instance['path_frame_end']
    src_node    = instance['node_seq'][0]
    dest_node   = instance['node_seq'][-1]

    # Look up the source port's actual start_frame_global in node_edge_df.
    # A ±20 frame tolerance covers any reward-fix shift in visit boundaries.
    src_cands = node_edge_df[
        (node_edge_df['location_type'] == 'node') &
        (node_edge_df['headstage_graph_node'] == src_node) &
        (node_edge_df['start_frame_global'] >= frame_start - 20) &
        (node_edge_df['start_frame_global'] <= frame_start + 20)
    ].sort_values('start_frame_global')
    ne_frame_start = (
        int(src_cands.iloc[0]['start_frame_global'])
        if not src_cands.empty else frame_start
    )

    # Look up the destination port's actual end_frame_global in node_edge_df.
    dest_cands = node_edge_df[
        (node_edge_df['location_type'] == 'node') &
        (node_edge_df['headstage_graph_node'] == dest_node) &
        (node_edge_df['start_frame_global'] >= frame_start) &
        (node_edge_df['start_frame_global'] <= frame_end + 20)
    ]
    ne_frame_end = (
        int(dest_cands.iloc[0]['end_frame_global'])
        if not dest_cands.empty else frame_end
    )

    sub = node_edge_df[
        (node_edge_df['start_frame_global'] >= ne_frame_start) &
        (node_edge_df['end_frame_global']   <= ne_frame_end)
    ].copy()

    types, labels = [], []
    for _, r in sub.iterrows():
        if r['location_type'] == 'node':
            types.append('node')
            labels.append(r['headstage_graph_node'])
        else:
            types.append('edge')
            labels.append(edge_label_str(r['headstage_graph_edge']))

    starts_abs    = sub['start_frame_global'].to_numpy() / FPS
    durations_sec = sub['duration'].to_numpy() / FPS
    ends_abs      = starts_abs + durations_sec
    return types, labels, starts_abs, ends_abs, durations_sec


for inst in direct_instances:
    types, labels, starts_abs_full, ends_abs_full, durations_full = (
        build_full_element_sequence(inst)
    )
    inst['full_seq_types']      = types
    inst['full_seq']            = labels
    inst['full_seq_starts_abs'] = starts_abs_full
    inst['full_seq_ends_abs']   = ends_abs_full
    inst['full_seq_durations']  = durations_full
    inst['n_full_elements']     = len(labels)

element_counts = [inst['n_full_elements'] for inst in direct_instances]
n_exactly_15 = sum(1 for c in element_counts if c == 15)
n_nodes_list  = [sum(t == 'node' for t in inst['full_seq_types']) for inst in direct_instances]
n_edges_list  = [sum(t == 'edge' for t in inst['full_seq_types']) for inst in direct_instances]

print(f"Built full node+edge sequences for {len(direct_instances)} direct-path instances")
print(f"Elements per instance : min={min(element_counts)}  max={max(element_counts)}  "
      f"mean={sum(element_counts)/len(element_counts):.1f}")
print(f"Node count per instance: min={min(n_nodes_list)}  max={max(n_nodes_list)}")
print(f"Edge count per instance: min={min(n_edges_list)}  max={max(n_edges_list)}")
print()
print(f"Instances with exactly 15 elements (9 nodes + 6 edges): "
      f"{n_exactly_15} / {len(direct_instances)}  "
      f"({100 * n_exactly_15 / len(direct_instances):.1f}%)")
if n_exactly_15 < len(direct_instances):
    from collections import Counter
    cnt = Counter(element_counts)
    other = {k: v for k, v in sorted(cnt.items()) if k != 15}
    print(f"Non-15 element counts: {other}")

Built full node+edge sequences for 426 direct-path instances
Elements per instance : min=5  max=47  mean=17.0
Node count per instance: min=4  max=25
Edge count per instance: min=1  max=22

Instances with exactly 15 elements (9 nodes + 6 edges): 119 / 426  (27.9%)
Non-15 element counts: {5: 1, 14: 46, 16: 70, 17: 56, 18: 28, 19: 35, 20: 25, 21: 8, 22: 12, 23: 9, 24: 8, 25: 3, 26: 4, 27: 1, 47: 1}


In [55]:
# Sanity check: collapsing the matched full sequence down to its node-only,
# consecutive-duplicates-merged entries should reproduce Step 1's node_seq
# exactly, because both node_summary_df and node_edge_df are generated by the
# same pipeline run (align_frames_trials.ipynb).
n_mismatch = 0
mismatch_detail = []

for inst in direct_instances:
    # Collapse full sequence → nodes only, remove consecutive duplicates
    collapsed_nodes = []
    for t, l in zip(inst['full_seq_types'], inst['full_seq']):
        if t == 'node' and (not collapsed_nodes or collapsed_nodes[-1] != l):
            collapsed_nodes.append(l)

    if collapsed_nodes != inst['node_seq']:
        n_mismatch += 1
        mismatch_detail.append({
            'pair': inst['pair_label'],
            'n_full':      inst['n_full_elements'],
            'collapsed':   collapsed_nodes,
            'node_seq':    inst['node_seq'],
        })

pct = 100 * n_mismatch / max(len(direct_instances), 1)
print(f"Instances where collapsed full-sequence ≠ node_seq: "
      f"{n_mismatch} / {len(direct_instances)}  ({pct:.1f}%)")

if n_mismatch == 0:
    print("✓ Perfect match — node_edge_df and node_summary_df are fully in sync.")
else:
    print(f"  (expected ~0 since both files come from the same pipeline)")
    print()
    for d in mismatch_detail[:5]:
        print(f"  pair={d['pair']}  n_full={d['n_full']}")
        print(f"    collapsed : {d['collapsed']}")
        print(f"    node_seq  : {d['node_seq']}")

# Also report element-count distribution among matched vs mismatched
from collections import Counter
matched_counts   = [inst['n_full_elements'] for inst in direct_instances
                    if inst['full_seq'] and [l for t, l in zip(inst['full_seq_types'], inst['full_seq']) if t == 'node'] == [l for t, l in zip(inst['full_seq_types'], inst['full_seq']) if t == 'node']]
cnt_15 = sum(1 for c in [inst['n_full_elements'] for inst in direct_instances] if c == 15)
print()
print(f"Element-count distribution: {dict(sorted(Counter(inst['n_full_elements'] for inst in direct_instances).items()))}")
print(f"  {cnt_15} instances have the expected 15 elements (9 nodes + 6 edges)")


Instances where collapsed full-sequence ≠ node_seq: 186 / 426  (43.7%)
  (expected ~0 since both files come from the same pipeline)

  pair=('Target5', 'Target2')  n_full=27
    collapsed : ['L521', 'L410', 'L521', 'L410', 'L521', 'L410', 'L521', 'L410', 'L35', 'L22', 'L11', 'L0', 'R0', 'R11', 'R23', 'R36', 'R413', 'R526']
    node_seq  : ['L521', 'L410', 'L35', 'L22', 'L11', 'L0', 'R0', 'R11', 'R23', 'R36', 'R413', 'R526']
  pair=('Target8', 'Target7')  n_full=18
    collapsed : ['R55', 'R42', 'R55', 'R42', 'R31', 'R20', 'R10', 'R21', 'R32', 'R44', 'R59']
    node_seq  : ['R55', 'R42', 'R31', 'R20', 'R10', 'R21', 'R32', 'R44', 'R59']
  pair=('Target5', 'Target4')  n_full=23
    collapsed : ['L521', 'L410', 'L521', 'L410', 'L35', 'L22', 'L11', 'L0', 'L10', 'L21', 'L32', 'L45', 'L510']
    node_seq  : ['L521', 'L410', 'L35', 'L22', 'L11', 'L0', 'L10', 'L21', 'L32', 'L45', 'L510']
  pair=('Target4', 'Target3')  n_full=15
    collapsed : ['L45', 'L32', 'L21', 'L10', 'L20', 'L31', 'L43', '

**Expected result:** With `node_edge_df.csv` generated by the same pipeline run as
`node_summary_df.csv`, the mismatch count above should be **0** (or very close to 0).

**Why 15 elements?**  
Every direct alternation-pair path traverses the same depth in the binary maze tree:
- 9 nodes: source port → 7 internal nodes → destination port  
- 6 physical edges: the 2 leaf-to-parent connections are *virtual* (no physical corridor)
  and produce no edge row in `node_edge_df`  
- Total: **15 elements**

**Port Y in consecutive paths:**  
For back-to-back direct paths X→Y and Y→Z, port Y appears in **both** sequences.  
The frame filter for X→Y is `start_frame ≥ start_frame[X]` AND `end_frame ≤ end_frame[Y]`,
and for Y→Z it is `start_frame ≥ start_frame[Y]` AND `end_frame ≤ end_frame[Z]`.  
Port Y's row satisfies both ranges exactly (boundary inclusivity on both sides).

**If mismatches remain:**  
Re-run `align_frames_trials.ipynb` to regenerate `node_edge_df.csv` and
`node_summary_df.csv` from the current traversal pipeline, then re-run this notebook.


## Per-element firing rate + canonical full sequence

Same approach as the per-node version: `compute_full_element_rates` gives one rate
value per element (node *or* edge) for one unit; `canonical_full_sequence` picks the
most common exact (type, label) sequence for a pair, used both as the x-axis labels
and the resampling target.

In [56]:
def canonical_full_sequence(instances):
    seqs = [tuple(zip(inst['full_seq_types'], inst['full_seq'])) for inst in instances]
    most_common_seq, _ = Counter(seqs).most_common(1)[0]
    types = [t for t, _ in most_common_seq]
    labels = [l for _, l in most_common_seq]
    return types, labels

def compute_full_element_rates(instance, cluster_id):
    """Firing rate (Hz) of one unit during each node/edge element of one path instance."""
    trials_in_path = trials_overlapping(instance['path_start_abs'], instance['path_end_abs'])
    rel_spikes = gather_relative_spikes(trials_in_path, cluster_id, instance['path_start_abs'])
    rel_starts = instance['full_seq_starts_abs'] - instance['path_start_abs']
    rel_ends = instance['full_seq_ends_abs'] - instance['path_start_abs']

    n = instance['n_full_elements']
    rates = np.empty(n)
    for i in range(n):
        count = np.sum((rel_spikes >= rel_starts[i]) & (rel_spikes < rel_ends[i]))
        rates[i] = count / instance['full_seq_durations'][i]
    return rates

## Aggregate mean +/- SEM per (pair, good unit), full node+edge version

For **alternation** pairs, all traversals are expected to follow the same canonical
node+edge sequence (9 nodes with physical edges between the 7 internal nodes and
virtual connections at source/target ports that produce no edge element). Rather than
resampling, each instance is matched to the canonical sequence by element identity
(greedy forward scan): positions where the instance is missing an element get NaN, so
the per-position mean and SEM only average instances that actually had that element.
Non-conforming instances (sequence differs from canonical) are counted and reported.

For **switch** pairs, paths have genuinely different canonical lengths across pairs, so
resampling onto the canonical grid is still used (same as before).

In [57]:
def align_to_canonical(instance, canon_types, canon_labels, cluster_id):
    """Per-canonical-position firing rate found by greedy forward name-matching.
    Canonical positions with no matching element in the instance are returned as NaN."""
    rates = compute_full_element_rates(instance, cluster_id)
    inst_types = instance['full_seq_types']
    inst_labels = instance['full_seq']
    L = len(canon_labels)
    row = np.full(L, np.nan)
    inst_idx = 0
    for canon_pos, (ctype, clabel) in enumerate(zip(canon_types, canon_labels)):
        for j in range(inst_idx, len(inst_labels)):
            if inst_labels[j] == clabel and inst_types[j] == ctype:
                row[canon_pos] = rates[j]
                inst_idx = j + 1
                break
    return row

full_pair_unit_stats = {}
skipped_full_pairs = []

for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        skipped_full_pairs.append((pair_label, len(instances)))
        continue

    canon_types, canon_labels = canonical_full_sequence(instances)
    L = len(canon_labels)
    path_class = instances[0]['path_class']

    canon_set = tuple(zip(canon_types, canon_labels))
    n_nonconforming = sum(
        1 for inst in instances
        if tuple(zip(inst['full_seq_types'], inst['full_seq'])) != canon_set
    )

    for cid in good_cluster_ids:
        if path_class == 'alternation':
            rows = np.vstack([
                align_to_canonical(inst, canon_types, canon_labels, cid)
                for inst in instances
            ])
            counts = np.sum(~np.isnan(rows), axis=0)
            mean = np.nanmean(rows, axis=0)
            sem = np.nanstd(rows, axis=0, ddof=1) / np.sqrt(np.maximum(counts, 1))
        else:
            rows = np.vstack([
                resample_to_grid(compute_full_element_rates(inst, cid), L)
                for inst in instances
            ])
            mean = rows.mean(axis=0)
            sem = rows.std(axis=0, ddof=1) / np.sqrt(rows.shape[0])

        full_pair_unit_stats[(pair_label, cid)] = {
            'mean': mean,
            'sem': sem,
            'n': len(instances),
            'n_nonconforming': n_nonconforming,
            'canon_types': canon_types,
            'canon_labels': canon_labels,
            'path_class': path_class,
        }

print(f"Computed full-sequence stats for {len(full_pair_unit_stats) // max(len(good_cluster_ids), 1)} pairs x {len(good_cluster_ids)} good units")
for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue
    stats0 = full_pair_unit_stats.get((pair_label, good_cluster_ids[0]))
    if stats0 and stats0['path_class'] == 'alternation' and stats0['n_nonconforming'] > 0:
        from_l, to_l = pair_label
        print(f"  {from_l}->{to_l}: {stats0['n_nonconforming']}/{len(instances)} instances deviate from canonical full sequence")

def _compute_full_pair_stats(split_dict):
    stats = {}
    for pair_label, instances in split_dict.items():
        if len(instances) < MIN_INSTANCES:
            continue
        canon_types, canon_labels = canonical_full_sequence(instances)
        L = len(canon_labels)
        path_class = instances[0]['path_class']
        canon_set = tuple(zip(canon_types, canon_labels))
        n_nonconforming = sum(
            1 for inst in instances
            if tuple(zip(inst['full_seq_types'], inst['full_seq'])) != canon_set
        )
        for cid in good_cluster_ids:
            if path_class == 'alternation':
                rows = np.vstack([
                    align_to_canonical(inst, canon_types, canon_labels, cid)
                    for inst in instances
                ])
                counts = np.sum(~np.isnan(rows), axis=0)
                mean = np.nanmean(rows, axis=0)
                sem = np.nanstd(rows, axis=0, ddof=1) / np.sqrt(np.maximum(counts, 1))
            else:
                rows = np.vstack([
                    resample_to_grid(compute_full_element_rates(inst, cid), L)
                    for inst in instances
                ])
                mean = rows.mean(axis=0)
                sem = rows.std(axis=0, ddof=1) / np.sqrt(rows.shape[0])
            stats[(pair_label, cid)] = {
                'mean': mean, 'sem': sem, 'n': len(instances),
                'n_nonconforming': n_nonconforming,
                'canon_types': canon_types, 'canon_labels': canon_labels,
                'path_class': path_class,
            }
    return stats

full_pair_unit_stats_rewarded   = _compute_full_pair_stats(pairs_to_rewarded_instances)
full_pair_unit_stats_unrewarded = _compute_full_pair_stats(pairs_to_unrewarded_instances)
n_rew   = len(full_pair_unit_stats_rewarded)   // max(len(good_cluster_ids), 1)
n_unrew = len(full_pair_unit_stats_unrewarded) // max(len(good_cluster_ids), 1)
print(f"Full stats — Rewarded pairs: {n_rew} | Unrewarded pairs: {n_unrew}")

Computed full-sequence stats for 22 pairs x 34 good units
  Target6->Target5: 31/36 instances deviate from canonical full sequence
  Target8->Target7: 39/52 instances deviate from canonical full sequence
  Target7->Target8: 36/53 instances deviate from canonical full sequence
  Target4->Target3: 43/56 instances deviate from canonical full sequence
  Target3->Target4: 35/46 instances deviate from canonical full sequence
  Target2->Target1: 38/47 instances deviate from canonical full sequence
  Target5->Target6: 29/38 instances deviate from canonical full sequence
  Target1->Target2: 22/30 instances deviate from canonical full sequence
Full stats — Rewarded pairs: 10 | Unrewarded pairs: 22


## Per-pair plots (full node+edge sequence)

X-axis is the canonical node+edge sequence (edge labels shown as `A-B`, italicized,
with a light gray background band, to visually separate transit elements from node
dwell elements). Saved under
`resources/outputs/{session}/direct_path_neural_with_edges/{alternation|switch}/{from}_to_{to}/unit_{cluster_id}.png`.

In [58]:
output_root_edges = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural_with_edges")
output_root_edges.mkdir(parents=True, exist_ok=True)

n_saved = 0
for pair_label, instances in pairs_to_instances.items():
    if len(instances) < MIN_INSTANCES:
        continue

    from_label, to_label = pair_label
    path_class = instances[0]['path_class']
    pair_dir = output_root_edges / path_class / f"{from_label}_to_{to_label}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    for cid in good_cluster_ids:
        c = TARGET_COLORS.get(to_label, 'steelblue')
        L_all = len(full_pair_unit_stats[(pair_label, cid)]['canon_labels'])

        fig, (ax_rew, ax_unrew) = plt.subplots(1, 2, figsize=(max(10, L_all * 0.8), 4))

        for ax, split_stats, split_label in [
            (ax_rew,   full_pair_unit_stats_rewarded.get((pair_label, cid)),   'Rewarded'),
            (ax_unrew, full_pair_unit_stats_unrewarded.get((pair_label, cid)), 'Unrewarded'),
        ]:
            ax.set_title(split_label, fontsize=9)

            if split_stats is None:
                ax.text(0.5, 0.5, f'< {MIN_INSTANCES} traversals', ha='center', va='center',
                        transform=ax.transAxes, fontsize=9, color='gray')
                ax.set_xticks([])
            else:
                mean, sem, n = split_stats['mean'], split_stats['sem'], split_stats['n']
                n_nc = split_stats['n_nonconforming']
                nc_note = f", {n_nc} non-conforming" if n_nc > 0 else ""
                ctypes = split_stats['canon_types']
                clabels = split_stats['canon_labels']
                x = np.arange(len(clabels))

                for i, t in enumerate(ctypes):
                    if t == 'edge':
                        ax.axvspan(i - 0.5, i + 0.5, color='lightgray', alpha=0.2, zorder=0)

                ax.plot(x, mean, color=c, linewidth=1.8, zorder=2)
                ax.fill_between(x, mean - sem, mean + sem, color=c, alpha=0.1, zorder=1,
                                label=f'SEM (n={n}{nc_note})')
                ax.set_xticks(x)
                ax.set_xticklabels(clabels, rotation=90, fontsize=6)
                for tick, t in zip(ax.get_xticklabels(), ctypes):
                    if t == 'edge':
                        tick.set_style('italic')
                        tick.set_color('dimgray')
                ax.legend(fontsize=7, loc='upper right')
            ax.set_xlabel('Path element (node = plain, edge = italic gray)')
            ax.set_ylabel('Firing rate (Hz)')

        fig.suptitle(
            f"Unit {cid} | {path_class}: {from_label} -> {to_label}",
            fontsize=9,
        )
        fig.tight_layout()
        fig.savefig(pair_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

print(f"Saved {n_saved} per-pair full-sequence plots under {output_root_edges}")

Saved 748 per-pair full-sequence plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_with_edges


## Alternation / Switch overview plots (full node+edge sequence)

Same normalized path-position overlay as the per-node overview, but built from the
full (node+edge) per-pair means -- pairs differ even more in length now that edges are
counted too. Saved under
`resources/outputs/{session}/direct_path_neural_with_edges/{alternation|switch}_overview/unit_{cluster_id}.png`.

In [59]:
for path_class in ['alternation', 'switch']:
    class_dir = output_root_edges / f"{path_class}_overview"
    class_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]

    # For alternation: node-indexed x-axis using most common canonical full-sequence length
    use_node_indexed = False
    L_expected = None
    node_x_vals = None
    alt_canon_types = None
    _mid = None
    _around_mid = []
    total_nonconforming = 0
    if path_class == 'alternation' and relevant_pairs:
        all_lengths = [len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels'])
                       for pl in relevant_pairs]
        L_expected = Counter(all_lengths).most_common(1)[0][0]
        _mid = (L_expected - 1) // 2
        node_x_vals = np.arange(L_expected)
        _around_mid = [_mid + d for d in [-4, -2, 2, 4] if 0 <= _mid + d < L_expected]
        for pl in relevant_pairs:
            if len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels']) == L_expected:
                alt_canon_types = full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_types']
                break
        total_nonconforming = sum(
            full_pair_unit_stats[(pl, good_cluster_ids[0])]['n_nonconforming']
            for pl in relevant_pairs
        )
        use_node_indexed = True

    # For switch, assign a linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    n_saved = 0
    for cid in good_cluster_ids:
        fig, (ax_rew, ax_unrew) = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

        for ax, split_stats_dict, split_label in [
            (ax_rew,   full_pair_unit_stats_rewarded,   'Rewarded'),
            (ax_unrew, full_pair_unit_stats_unrewarded, 'Unrewarded'),
        ]:
            ax.set_title(split_label, fontsize=10)

            if use_node_indexed and alt_canon_types is not None:
                for i, t in enumerate(alt_canon_types):
                    if t == 'edge':
                        ax.axvspan(i - 0.5, i + 0.5, color='lightgray', alpha=0.2, zorder=0)

            for pair_label in relevant_pairs:
                if (pair_label, cid) not in split_stats_dict:
                    continue
                stats = split_stats_dict[(pair_label, cid)]
                from_label, to_label = pair_label
                color = TARGET_COLORS.get(to_label, 'gray')
                ls = from_to_ls.get(from_label, '-')
                L = len(stats['canon_labels'])

                if use_node_indexed:
                    if L == L_expected:
                        mean_plot, sem_plot = stats['mean'], stats['sem']
                    else:
                        mean_plot = resample_to_grid(stats['mean'], L_expected)
                        sem_plot  = resample_to_grid(stats['sem'],  L_expected)
                    x_plot = node_x_vals
                else:
                    mean_plot = resample_to_grid(stats['mean'], OVERVIEW_GRID) if L != OVERVIEW_GRID else stats['mean']
                    sem_plot  = resample_to_grid(stats['sem'],  OVERVIEW_GRID) if L != OVERVIEW_GRID else stats['sem']
                    x_plot = frac_grid

                ax.plot(x_plot, mean_plot, color=color, linewidth=1.5, linestyle=ls,
                         label=f"{from_label}->{to_label} (n={stats['n']})")
                ax.fill_between(x_plot, mean_plot - sem_plot, mean_plot + sem_plot,
                                 color=color, alpha=0.1)

            if use_node_indexed:
                tick_positions = sorted(set([0, *_around_mid, _mid, L_expected - 1]))
                label_at = {0: 'Source\nPort', _mid: '', L_expected - 1: 'Target\nPort'}
                ax.set_xticks(tick_positions)
                ax.set_xticklabels([label_at.get(p, '') for p in tick_positions])
                for tick, pos in zip(ax.get_xticklabels(), tick_positions):
                    if pos in (0, L_expected - 1):
                        tick.set_color('#27ae60')
                        tick.set_fontweight('bold')
                ax.set_xlabel('Path element position (gray = edge transit)')
                trans = ax.get_xaxis_transform()
                for pos in _around_mid:
                    ax.plot([pos], [0], 'o', color='#e67e22', markersize=9,
                            markeredgecolor='white', markeredgewidth=0.8,
                            transform=trans, clip_on=False, zorder=5)
                ax.plot([_mid], [0], 'o', color='#2980b9', markersize=11,
                        markeredgecolor='#c0392b', markeredgewidth=2.0,
                        transform=trans, clip_on=False, zorder=6)
            else:
                ax.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')
            ax.legend(fontsize=6, loc='upper right', ncol=2)

        ax_rew.set_ylabel('Firing rate (Hz)')
        nc_note = f" | {total_nonconforming} non-conforming" if total_nonconforming > 0 else ""
        fig.suptitle(f"Unit {cid} | {path_class.capitalize()} direct paths (node+edge){nc_note}", fontsize=10)
        fig.tight_layout()
        fig.savefig(class_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} '{path_class}' full-sequence overview plots under {class_dir} ({len(relevant_pairs)} pairs overlaid)")

Saved 34 'alternation' full-sequence overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_with_edges/alternation_overview (8 pairs overlaid)
Saved 34 'switch' full-sequence overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_with_edges/switch_overview (14 pairs overlaid)


## Save full-sequence pair summary

In [60]:
full_summary_rows = []
for pair_label, instances in pairs_to_instances.items():
    from_label, to_label = pair_label
    canon_types, canon_labels = canonical_full_sequence(instances)
    full_summary_rows.append({
        'from_port': from_label,
        'to_port': to_label,
        'path_class': instances[0]['path_class'],
        'n_instances': len(instances),
        'n_full_elements_canonical': len(canon_labels),
        'n_nodes_canonical': len(instances[0]['node_seq']),
        'used_in_aggregation': len(instances) >= MIN_INSTANCES,
        'canonical_full_sequence': ' -> '.join(canon_labels),
    })

full_path_pair_summary_df = pd.DataFrame(full_summary_rows).sort_values('n_instances', ascending=False).reset_index(drop=True)
out_csv = output_root_edges / 'direct_path_pair_summary_with_edges.csv'
full_path_pair_summary_df.to_csv(out_csv, index=False)
print(f"Saved full-sequence pair summary to {out_csv}")
display(full_path_pair_summary_df)

Saved full-sequence pair summary to /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_with_edges/direct_path_pair_summary_with_edges.csv


,from_port,to_port,path_class,n_instances,n_full_elements_canonical,n_nodes_canonical,used_in_aggregation,canonical_full_sequence
0,Target4,Target3,alternation,56,15,9,True,L510 -> L45 -> L32-L45 -> L32 -> L21-L32 -> L2...
1,Target7,Target8,alternation,53,15,9,True,R59 -> R44 -> R32-R44 -> R32 -> R21-R32 -> R21...
2,Target8,Target7,alternation,52,15,9,True,R55 -> R42 -> R31-R42 -> R31 -> R20-R31 -> R20...
3,Target2,Target1,alternation,47,15,9,True,R526 -> R413 -> R36-R413 -> R36 -> R23-R36 -> ...
4,Target3,Target4,alternation,46,15,9,True,L56 -> L43 -> L31-L43 -> L31 -> L20-L31 -> L20...
5,Target5,Target6,alternation,38,14,9,True,L410 -> L35-L410 -> L35 -> L22-L35 -> L22 -> L...
6,Target6,Target5,alternation,36,15,9,True,L525 -> L412 -> L36-L412 -> L36 -> L23-L36 -> ...
7,Target1,Target2,alternation,30,15,9,True,R522 -> R411 -> R35-R411 -> R35 -> R22-R35 -> ...
8,Target2,Target8,switch,12,19,11,True,R526 -> R413 -> R36-R413 -> R36 -> R23-R36 -> ...
9,Target5,Target4,switch,7,19,11,True,L521 -> L410 -> L35-L410 -> L35 -> L22-L35 -> ...


# Combined 3-panel overview: continuous / node+edge / node-only

One figure per good unit per path class with three stacked panels:

- **Top**: continuous-time binned firing rate (normalized path time)
- **Middle**: node + edge sequence firing rate (element-position x-axis for alternation, gray bands = edge transits)
- **Bottom**: node-only firing rate (node-position x-axis for alternation)

For alternation the middle and bottom x-axes are structural (Source Port → Mid Node → Target Port). For switch they stay normalized 0–1. The figure title notes the total number of non-conforming alternation instances across all pairs.

In [61]:
combined_output_root = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural_combined")
combined_output_root.mkdir(parents=True, exist_ok=True)

for path_class in ['alternation', 'switch']:
    combined_dir = combined_output_root / f"{path_class}_overview"
    combined_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]
    if not relevant_pairs:
        continue

    # For switch: linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    # Node-indexed x-axis setup for alternation
    use_node_indexed = False
    L_nodes_exp = L_full_exp = mid_node = mid_full = None
    alt_full_canon_types = None
    _around_mid_full = []
    _around_mid_node = []
    total_nc = 0
    if path_class == 'alternation':
        node_lengths = [len(pair_unit_stats[(pl, good_cluster_ids[0])]['canonical_seq'])
                        for pl in relevant_pairs]
        L_nodes_exp = Counter(node_lengths).most_common(1)[0][0]
        mid_node = (L_nodes_exp - 1) // 2

        full_lengths = [len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels'])
                        for pl in relevant_pairs]
        L_full_exp = Counter(full_lengths).most_common(1)[0][0]
        mid_full = (L_full_exp - 1) // 2

        for pl in relevant_pairs:
            if len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels']) == L_full_exp:
                alt_full_canon_types = full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_types']
                break

        _around_mid_full = [mid_full + d for d in [-4, -2, 2, 4] if 0 <= mid_full + d < L_full_exp]
        _around_mid_node = [mid_node + d for d in [-2, -1, 1, 2] if 0 <= mid_node + d < L_nodes_exp]
        total_nc = sum(
            full_pair_unit_stats[(pl, good_cluster_ids[0])]['n_nonconforming']
            for pl in relevant_pairs
        )
        use_node_indexed = True

    reward_splits = [
        ('Rewarded',   continuous_pair_unit_stats_rewarded,   full_pair_unit_stats_rewarded,   pair_unit_stats_rewarded),
        ('Unrewarded', continuous_pair_unit_stats_unrewarded, full_pair_unit_stats_unrewarded, pair_unit_stats_unrewarded),
    ]

    n_saved = 0
    for cid in good_cluster_ids:
        fig, axes = plt.subplots(3, 2, figsize=(16, 11), sharey='row')

        legend_handles, legend_labels_list = [], []

        for col_idx, (split_label, cont_dict, full_dict, node_dict) in enumerate(reward_splits):
            ax_cont = axes[0, col_idx]
            ax_full = axes[1, col_idx]
            ax_node = axes[2, col_idx]

            ax_cont.set_title(f'Continuous — {split_label}', fontsize=9)
            ax_full.set_title(f'Node + edge — {split_label}', fontsize=9)
            ax_node.set_title(f'Node only — {split_label}', fontsize=9)

            if use_node_indexed and alt_full_canon_types is not None:
                for i, t in enumerate(alt_full_canon_types):
                    if t == 'edge':
                        ax_full.axvspan(i - 0.5, i + 0.5, color='lightgray', alpha=0.2, zorder=0)

            for pair_label in relevant_pairs:
                from_label, to_label = pair_label
                color = TARGET_COLORS.get(to_label, 'gray')
                ls = from_to_ls.get(from_label, '-')

                # ── Continuous ───────────────────────────────────────────────────────
                if (pair_label, cid) in cont_dict:
                    sc = cont_dict[(pair_label, cid)]
                    line, = ax_cont.plot(continuous_frac_grid, sc['mean'], color=color, linewidth=1.5, linestyle=ls)
                    ax_cont.fill_between(continuous_frac_grid, sc['mean'] - sc['sem'],
                                          sc['mean'] + sc['sem'], color=color, alpha=0.1)
                    if col_idx == 0:
                        legend_handles.append(line)
                        legend_labels_list.append(f"{from_label}->{to_label} (n={sc['n']})")

                # ── Node+edge ─────────────────────────────────────────────────────────
                if (pair_label, cid) in full_dict:
                    sf = full_dict[(pair_label, cid)]
                    L_f = len(sf['canon_labels'])
                    if use_node_indexed:
                        mf   = sf['mean'] if L_f == L_full_exp else resample_to_grid(sf['mean'], L_full_exp)
                        semf = sf['sem']  if L_f == L_full_exp else resample_to_grid(sf['sem'],  L_full_exp)
                        xf   = np.arange(L_full_exp)
                    else:
                        mf   = resample_to_grid(sf['mean'], OVERVIEW_GRID) if L_f != OVERVIEW_GRID else sf['mean']
                        semf = resample_to_grid(sf['sem'],  OVERVIEW_GRID) if L_f != OVERVIEW_GRID else sf['sem']
                        xf   = frac_grid
                    ax_full.plot(xf, mf, color=color, linewidth=1.5, linestyle=ls)
                    ax_full.fill_between(xf, mf - semf, mf + semf, color=color, alpha=0.1)

                # ── Node-only ─────────────────────────────────────────────────────────
                if (pair_label, cid) in node_dict:
                    sn = node_dict[(pair_label, cid)]
                    L_n = len(sn['canonical_seq'])
                    if use_node_indexed:
                        mn   = sn['mean'] if L_n == L_nodes_exp else resample_to_grid(sn['mean'], L_nodes_exp)
                        semn = sn['sem']  if L_n == L_nodes_exp else resample_to_grid(sn['sem'],  L_nodes_exp)
                        xn   = np.arange(L_nodes_exp)
                    else:
                        mn   = resample_to_grid(sn['mean'], OVERVIEW_GRID) if L_n != OVERVIEW_GRID else sn['mean']
                        semn = resample_to_grid(sn['sem'],  OVERVIEW_GRID) if L_n != OVERVIEW_GRID else sn['sem']
                        xn   = frac_grid
                    ax_node.plot(xn, mn, color=color, linewidth=1.5, linestyle=ls)
                    ax_node.fill_between(xn, mn - semn, mn + semn, color=color, alpha=0.1)

            # ── Format axes ───────────────────────────────────────────────────────────
            ax_cont.set_xlabel('Normalized path time (0 = source port start, 1 = target port end)')
            if col_idx == 0:
                ax_cont.set_ylabel('Firing rate (Hz)')
                ax_full.set_ylabel('Firing rate (Hz)')
                ax_node.set_ylabel('Firing rate (Hz)')

            if use_node_indexed:
                tick_positions_full = sorted(set([0, *_around_mid_full, mid_full, L_full_exp - 1]))
                label_at_full = {0: 'Source\nPort', mid_full: '', L_full_exp - 1: 'Target\nPort'}
                ax_full.set_xticks(tick_positions_full)
                ax_full.set_xticklabels([label_at_full.get(p, '') for p in tick_positions_full])
                for tick, pos in zip(ax_full.get_xticklabels(), tick_positions_full):
                    if pos in (0, L_full_exp - 1):
                        tick.set_color('#27ae60')
                        tick.set_fontweight('bold')
                ax_full.set_xlabel('Path element position (gray = edge transit)')

                tick_positions_node = sorted(set([0, *_around_mid_node, mid_node, L_nodes_exp - 1]))
                label_at_node = {0: 'Source\nPort', mid_node: '', L_nodes_exp - 1: 'Target\nPort'}
                ax_node.set_xticks(tick_positions_node)
                ax_node.set_xticklabels([label_at_node.get(p, '') for p in tick_positions_node])
                for tick, pos in zip(ax_node.get_xticklabels(), tick_positions_node):
                    if pos in (0, L_nodes_exp - 1):
                        tick.set_color('#27ae60')
                        tick.set_fontweight('bold')
                ax_node.set_xlabel('Path node position')

                trans_full = ax_full.get_xaxis_transform()
                for pos in _around_mid_full:
                    ax_full.plot([pos], [0], 'o', color='#e67e22', markersize=9,
                                markeredgecolor='white', markeredgewidth=0.8,
                                transform=trans_full, clip_on=False, zorder=5)
                ax_full.plot([mid_full], [0], 'o', color='#2980b9', markersize=11,
                            markeredgecolor='#c0392b', markeredgewidth=2.0,
                            transform=trans_full, clip_on=False, zorder=6)

                trans_node = ax_node.get_xaxis_transform()
                for pos in _around_mid_node:
                    ax_node.plot([pos], [0], 'o', color='#e67e22', markersize=9,
                                markeredgecolor='white', markeredgewidth=0.8,
                                transform=trans_node, clip_on=False, zorder=5)
                ax_node.plot([mid_node], [0], 'o', color='#2980b9', markersize=11,
                            markeredgecolor='#c0392b', markeredgewidth=2.0,
                            transform=trans_node, clip_on=False, zorder=6)
            else:
                ax_full.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')
                ax_node.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')

        nc_note = f" | {total_nc} non-conforming" if total_nc > 0 else ""
        fig.suptitle(f"Unit {cid} | {path_class.capitalize()} direct paths{nc_note}",
                     fontsize=11, fontweight='bold')

        n_cols = min(len(legend_labels_list), 4)
        fig.tight_layout()
        fig.subplots_adjust(bottom=0.09, top=0.94, hspace=0.55)
        fig.legend(legend_handles, legend_labels_list,
                   loc='lower center', bbox_to_anchor=(0.5, 0),
                   fontsize=6, ncol=n_cols, frameon=True)

        fig.savefig(combined_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} combined '{path_class}' overview plots under {combined_dir}")

Saved 34 combined 'alternation' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_combined/alternation_overview
Saved 34 combined 'switch' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_combined/switch_overview


# Alternation average plot: mean of all paths, continuous + node+edge

One figure per good unit showing the **grand mean** (averaged across all alternation
pairs) in two panels:
- **Top**: continuous-time firing rate (same grid as the continuous analysis)
- **Bottom**: full node+edge sequence with gray edge shading and annotated x-axis:
  - source / target port labels in **green**
  - midpoint marked with a **blue circle + red outline**
  - ±1 and ±2 positions around the midpoint marked with **orange circles**

Saved under `resources/outputs/{session}/direct_path_neural_alternation_avg/`.

In [62]:
# ── Alternation grand-mean plot: average across all paths, continuous + node+edge ──

avg_output_root = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural_alternation_avg")
avg_output_root.mkdir(parents=True, exist_ok=True)

alt_pairs = [
    pl for pl, insts in pairs_to_instances.items()
    if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == 'alternation'
]

if not alt_pairs:
    print("No alternation pairs found — no plots generated.")
else:
    # Canonical full-sequence length (most common across pairs)
    full_lengths = [len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels'])
                    for pl in alt_pairs]
    L_full_exp = Counter(full_lengths).most_common(1)[0][0]
    mid_full = (L_full_exp - 1) // 2

    alt_full_canon_types = None
    for pl in alt_pairs:
        if len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels']) == L_full_exp:
            alt_full_canon_types = full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_types']
            break

    _around_mid = [mid_full + d for d in [-4, -2, 2, 4] if 0 <= mid_full + d < L_full_exp]

    def _resamp_full(pl, stats_dict):
        sf = stats_dict.get((pl, cid))
        if sf is None:
            return None
        m = sf['mean']
        return m if len(m) == L_full_exp else resample_to_grid(m, L_full_exp)

    def _grand_mean(pl_list, stats_dict_cont, stats_dict_full):
        cont_rows = [stats_dict_cont[(pl, cid)]['mean'] for pl in pl_list
                     if (pl, cid) in stats_dict_cont]
        full_rows = [r for pl in pl_list for r in [_resamp_full(pl, stats_dict_full)] if r is not None]
        if not cont_rows:
            return None, None, None, None
        cont_arr = np.vstack(cont_rows)
        full_arr = np.vstack(full_rows) if full_rows else None
        mean_cont = cont_arr.mean(axis=0)
        sem_cont  = cont_arr.std(axis=0, ddof=1) / np.sqrt(len(cont_rows)) if len(cont_rows) > 1 else np.zeros_like(mean_cont)
        if full_arr is not None and len(full_arr):
            mean_full = full_arr.mean(axis=0)
            sem_full  = full_arr.std(axis=0, ddof=1) / np.sqrt(len(full_arr)) if len(full_arr) > 1 else np.zeros_like(mean_full)
        else:
            mean_full = sem_full = None
        return mean_cont, sem_cont, mean_full, sem_full

    n_saved = 0
    for cid in good_cluster_ids:
        reward_splits = [
            ('Rewarded',   continuous_pair_unit_stats_rewarded,   full_pair_unit_stats_rewarded,   [pl for pl in alt_pairs if (pl, cid) in continuous_pair_unit_stats_rewarded]),
            ('Unrewarded', continuous_pair_unit_stats_unrewarded, full_pair_unit_stats_unrewarded, [pl for pl in alt_pairs if (pl, cid) in continuous_pair_unit_stats_unrewarded]),
        ]

        fig, axes = plt.subplots(2, 2, figsize=(14, 8))
        (ax_cont_rew, ax_cont_unrew), (ax_full_rew, ax_full_unrew) = axes

        for col_idx, (split_label, cont_dict, full_dict, split_alt_pairs) in enumerate(reward_splits):
            ax_cont = axes[0, col_idx]
            ax_full = axes[1, col_idx]
            ax_cont.set_title(f'Continuous — {split_label}', fontsize=9)
            ax_full.set_title(f'Node + edge — {split_label}', fontsize=9)

            if not split_alt_pairs:
                for ax in (ax_cont, ax_full):
                    ax.text(0.5, 0.5, f'< {MIN_INSTANCES} traversals', ha='center', va='center',
                            transform=ax.transAxes, fontsize=9, color='gray')
                continue

            mean_cont, sem_cont, mean_full, sem_full = _grand_mean(split_alt_pairs, cont_dict, full_dict)

            if mean_cont is not None:
                ax_cont.plot(continuous_frac_grid, mean_cont, color='steelblue', linewidth=2.0)
                ax_cont.fill_between(continuous_frac_grid, mean_cont - sem_cont, mean_cont + sem_cont,
                                     color='steelblue', alpha=0.15)
            ax_cont.set_xlabel('Normalized path time (0 = source port, 1 = target port)')
            if col_idx == 0:
                ax_cont.set_ylabel('Firing rate (Hz)')

            if alt_full_canon_types is not None:
                for i, t in enumerate(alt_full_canon_types):
                    if t == 'edge':
                        ax_full.axvspan(i - 0.5, i + 0.5, color='lightgray', alpha=0.2, zorder=0)

            if mean_full is not None:
                x_full = np.arange(L_full_exp)
                ax_full.plot(x_full, mean_full, color='steelblue', linewidth=2.0, zorder=2)
                ax_full.fill_between(x_full, mean_full - sem_full, mean_full + sem_full,
                                     color='steelblue', alpha=0.15, zorder=1)

            tick_positions = sorted(set([0, *_around_mid, mid_full, L_full_exp - 1]))
            label_at = {0: 'Source\nPort', mid_full: '', L_full_exp - 1: 'Target\nPort'}
            ax_full.set_xticks(tick_positions)
            ax_full.set_xticklabels([label_at.get(p, '') for p in tick_positions])
            for tick, pos in zip(ax_full.get_xticklabels(), tick_positions):
                if pos in (0, L_full_exp - 1):
                    tick.set_color('#27ae60')
                    tick.set_fontweight('bold')
            if col_idx == 0:
                ax_full.set_ylabel('Firing rate (Hz)')
            ax_full.set_xlabel('Path element position (gray = edge transit)')

            trans = ax_full.get_xaxis_transform()
            for pos in _around_mid:
                ax_full.plot([pos], [0], 'o', color='#e67e22', markersize=9,
                            markeredgecolor='white', markeredgewidth=0.8,
                            transform=trans, clip_on=False, zorder=5)
            ax_full.plot([mid_full], [0], 'o', color='#2980b9', markersize=11,
                        markeredgecolor='#c0392b', markeredgewidth=2.0,
                        transform=trans, clip_on=False, zorder=6)

        fig.suptitle(
            f"Unit {cid} | Alternation direct paths — grand mean of {len(alt_pairs)} paths",
            fontsize=11, fontweight='bold',
        )
        fig.tight_layout()
        fig.subplots_adjust(bottom=0.1)
        fig.savefig(avg_output_root / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} alternation-average plots under {avg_output_root} ({len(alt_pairs)} paths averaged)")

Saved 34 alternation-average plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_alternation_avg (8 paths averaged)


In [63]:
cont_trav_output_root = Path(f"/var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/{DD}_{MM}_{YYYY}/direct_path_neural_continous+traversal")
cont_trav_output_root.mkdir(parents=True, exist_ok=True)

for path_class in ['alternation', 'switch']:
    cont_trav_dir = cont_trav_output_root / f"{path_class}_overview"
    cont_trav_dir.mkdir(parents=True, exist_ok=True)

    relevant_pairs = [
        pl for pl, insts in pairs_to_instances.items()
        if len(insts) >= MIN_INSTANCES and insts[0]['path_class'] == path_class
    ]
    if not relevant_pairs:
        continue

    # For switch: linestyle per unique from_label
    from_to_ls = {}
    if path_class == 'switch':
        unique_from_labels = sorted(set(pl[0] for pl in relevant_pairs))
        from_to_ls = {fl: SWITCH_FROM_LINESTYLES[i % len(SWITCH_FROM_LINESTYLES)]
                      for i, fl in enumerate(unique_from_labels)}

    # Node-indexed x-axis setup for alternation
    use_node_indexed = False
    L_full_exp = mid_full = None
    alt_full_canon_types = None
    _around_mid_full = []
    total_nc = 0
    if path_class == 'alternation':
        full_lengths = [len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels'])
                        for pl in relevant_pairs]
        L_full_exp = Counter(full_lengths).most_common(1)[0][0]
        mid_full = (L_full_exp - 1) // 2

        for pl in relevant_pairs:
            if len(full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_labels']) == L_full_exp:
                alt_full_canon_types = full_pair_unit_stats[(pl, good_cluster_ids[0])]['canon_types']
                break

        _around_mid_full = [mid_full + d for d in [-4, -2, 2, 4] if 0 <= mid_full + d < L_full_exp]
        total_nc = sum(
            full_pair_unit_stats[(pl, good_cluster_ids[0])]['n_nonconforming']
            for pl in relevant_pairs
        )
        use_node_indexed = True

    reward_splits = [
        ('Rewarded',   continuous_pair_unit_stats_rewarded,   full_pair_unit_stats_rewarded),
        ('Unrewarded', continuous_pair_unit_stats_unrewarded, full_pair_unit_stats_unrewarded),
    ]

    n_saved = 0
    for cid in good_cluster_ids:
        fig, axes = plt.subplots(2, 2, figsize=(16, 11), sharey='row')

        legend_handles, legend_labels_list = [], []

        for col_idx, (split_label, cont_dict, full_dict) in enumerate(reward_splits):
            ax_cont = axes[0, col_idx]
            ax_full = axes[1, col_idx]

            ax_cont.set_title(f'Continuous — {split_label}', fontsize=14)
            ax_full.set_title(f'Node + edge — {split_label}', fontsize=14)

            if use_node_indexed and alt_full_canon_types is not None:
                for i, t in enumerate(alt_full_canon_types):
                    if t == 'edge':
                        ax_full.axvspan(i - 0.5, i + 0.5, color='lightgray', alpha=0.2, zorder=0)

            for pair_label in relevant_pairs:
                from_label, to_label = pair_label
                color = TARGET_COLORS.get(to_label, 'gray')
                ls = from_to_ls.get(from_label, '-')

                # ── Continuous ───────────────────────────────────────────────────────
                if (pair_label, cid) in cont_dict:
                    sc = cont_dict[(pair_label, cid)]
                    line, = ax_cont.plot(continuous_frac_grid, sc['mean'], color=color, linewidth=1.5, linestyle=ls)
                    ax_cont.fill_between(continuous_frac_grid, sc['mean'] - sc['sem'],
                                          sc['mean'] + sc['sem'], color=color, alpha=0.1)
                    if col_idx == 0:
                        legend_handles.append(line)
                        legend_labels_list.append(f"{from_label}->{to_label} (n={sc['n']})")

                # ── Node+edge ─────────────────────────────────────────────────────────
                if (pair_label, cid) in full_dict:
                    sf = full_dict[(pair_label, cid)]
                    L_f = len(sf['canon_labels'])
                    if use_node_indexed:
                        mf   = sf['mean'] if L_f == L_full_exp else resample_to_grid(sf['mean'], L_full_exp)
                        semf = sf['sem']  if L_f == L_full_exp else resample_to_grid(sf['sem'],  L_full_exp)
                        xf   = np.arange(L_full_exp)
                    else:
                        mf   = resample_to_grid(sf['mean'], OVERVIEW_GRID) if L_f != OVERVIEW_GRID else sf['mean']
                        semf = resample_to_grid(sf['sem'],  OVERVIEW_GRID) if L_f != OVERVIEW_GRID else sf['sem']
                        xf   = frac_grid
                    ax_full.plot(xf, mf, color=color, linewidth=1.5, linestyle=ls)
                    ax_full.fill_between(xf, mf - semf, mf + semf, color=color, alpha=0.1)

            # ── Format axes ───────────────────────────────────────────────────────────
            ax_cont.set_xlabel('Normalized path time (0 = source port start, 1 = target port end)')
            if col_idx == 0:
                ax_cont.set_ylabel('Firing rate (Hz)')
                ax_full.set_ylabel('Firing rate (Hz)')

            if use_node_indexed:
                tick_positions_full = sorted(set([0, *_around_mid_full, mid_full, L_full_exp - 1]))
                label_at_full = {0: 'Source\nPort', mid_full: '', L_full_exp - 1: 'Target\nPort'}
                ax_full.set_xticks(tick_positions_full)
                ax_full.set_xticklabels([label_at_full.get(p, '') for p in tick_positions_full])
                for tick, pos in zip(ax_full.get_xticklabels(), tick_positions_full):
                    if pos in (0, L_full_exp - 1):
                        tick.set_color('#27ae60')
                        tick.set_fontweight('bold')
                ax_full.set_xlabel('Path element position (gray = edge transit)')

                trans_full = ax_full.get_xaxis_transform()
                for pos in _around_mid_full:
                    ax_full.plot([pos], [0], 'o', color='#e67e22', markersize=9,
                                markeredgecolor='white', markeredgewidth=0.8,
                                transform=trans_full, clip_on=False, zorder=5)
                ax_full.plot([mid_full], [0], 'o', color='#2980b9', markersize=11,
                            markeredgecolor='#c0392b', markeredgewidth=2.0,
                            transform=trans_full, clip_on=False, zorder=6)
            else:
                ax_full.set_xlabel('Normalized path position (0 = source port start, 1 = target port end)')

        nc_note = f" | {total_nc} non-conforming" if total_nc > 0 else ""
        fig.suptitle(f"Unit {cid} | {path_class.capitalize()} direct paths{nc_note}",
                     fontsize=16, fontweight='bold')

        n_cols = min(len(legend_labels_list), 4)
        fig.tight_layout()
        fig.subplots_adjust(bottom=0.09, top=0.94, hspace=0.45)
        fig.legend(legend_handles, legend_labels_list,
                   loc='lower center', bbox_to_anchor=(0.5, 0),
                   fontsize=12, ncol=n_cols, frameon=True)

        fig.savefig(cont_trav_dir / f"unit_{cid:04d}_good.png", dpi=150)
        plt.close(fig)
        n_saved += 1

    print(f"Saved {n_saved} combined '{path_class}' overview plots under {cont_trav_dir}")

Saved 34 combined 'alternation' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_continous+traversal/alternation_overview
Saved 34 combined 'switch' overview plots under /var/home/almogmeir/Documents/GitHub/NaviGraph/examples/dual_root_tree/resources/outputs/10_09_2025/direct_path_neural_continous+traversal/switch_overview
